# Random-Vocab Long-Range Null on Llama-3.1-8B

Closes the remaining loose end on the long-range power-law claim. Uses the **same log-spaced protocol** as `Corpus_Expansion_LongRange_Llama` (context lengths [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]) but with **random-vocab "documents"** instead of natural text.

**Two variants:**
- `random_vocab_uniform`: uniform sampling from Llama vocab.
- `random_vocab_freq`: sampling from natural-corpus token-frequency distribution (gutenberg_fiction_en).

**Prediction:** the per-token order-specific gap should be ≈ 0 at every distance (or small and flat), confirming that the slope-≈-1 power law observed in natural cells is genuinely a property of natural ordered text, not the long-range measurement protocol or the probe responding to any input.

**Output:** `My Drive/LRTIA/Results/corpus_expansion_longrange/llama/random_vocab_<variant>.json` — same dir and format as the natural-language long-range cells.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, random, time
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/LRTIA')
BASE = DRIVE / 'Results/corpus_expansion_longrange/llama'
BASE.mkdir(parents=True, exist_ok=True)
FREQ_SOURCE_TXT = DRIVE / 'Data/corpus_expansion/gutenberg_fiction_en'

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'

# Same protocol as Corpus_Expansion_LongRange_Llama.
CTX_LENGTHS = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
MAX_CTX = max(CTX_LENGTHS)
TARGET_LEN = 30
DOC_TOK_LEN = MAX_CTX + TARGET_LEN + 50  # 1104 tokens per synthetic doc
DOCS_PER_VARIANT = 60
TARGET_FRACS = [0.5]
N_SHUFFLES = 1
SEED = 20260503

RUN_VARIANTS = ['random_vocab_uniform', 'random_vocab_freq']

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Probe: {MODEL_NAME}')
print(f'Context lengths: {CTX_LENGTHS}')
print(f'Per variant: {DOCS_PER_VARIANT} docs × {DOC_TOK_LEN} tokens × 1 target')

In [ ]:
# Load model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
VOCAB_SIZE = tokenizer.vocab_size
print(f'{MODEL_NAME} loaded. vocab_size = {VOCAB_SIZE}')

In [ ]:
# Pipeline — identical to the natural-language long-range notebook so cache files merge cleanly.
@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2:
        return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i + 1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf'), float('inf')
    mn = nll / cnt
    return math.exp(mn), mn

def compute_longrange_curves(full_ids, target_start, target_end):
    tgt = full_ids[target_start:target_end]
    o_ppl, s_ppl = [], []
    for c in CTX_LENGTHS:
        if c == 0:
            pfx = []
        else:
            pfx = full_ids[target_start - c:target_start]
        p_ord, _ = ppl_nll(pfx, tgt)
        o_ppl.append(p_ord)
        if c == 0:
            s_ppl.append(p_ord)
        else:
            rng = random.Random(SEED + c)
            sh_ppls = []
            for _ in range(N_SHUFFLES):
                sh = list(pfx); rng.shuffle(sh)
                p_sh, _ = ppl_nll(sh, tgt)
                if not math.isinf(p_sh): sh_ppls.append(p_sh)
            s_ppl.append(np.mean(sh_ppls) if sh_ppls else p_ord)
    return {
        'context_lengths': list(CTX_LENGTHS),
        'ordered_ppl': o_ppl,
        'shuffled_ppl': s_ppl,
    }

print('Pipeline ready')

In [ ]:
# Build the freq distribution from gutenberg fiction (same as the dense random_vocab run).
freq_counter = Counter()
src_files = list(FREQ_SOURCE_TXT.glob('*.txt'))[:30]
for fp in tqdm(src_files, desc='counting tokens'):
    text = fp.read_text(encoding='utf-8', errors='replace')
    ids = tokenizer.encode(text, add_special_tokens=False)
    freq_counter.update(ids)
freq_ids = np.array(list(freq_counter.keys()), dtype=np.int64)
freq_weights = np.array(list(freq_counter.values()), dtype=np.float64)
freq_weights = freq_weights / freq_weights.sum()
print(f'Built freq distribution from {len(src_files)} files: {len(freq_ids)} unique tokens')

In [ ]:
def gen_uniform_doc(rng):
    return rng.integers(0, VOCAB_SIZE, size=DOC_TOK_LEN, dtype=np.int64).tolist()

def gen_freq_doc(rng):
    idx = rng.choice(len(freq_ids), size=DOC_TOK_LEN, p=freq_weights)
    return freq_ids[idx].tolist()

GEN = {
    'random_vocab_uniform': gen_uniform_doc,
    'random_vocab_freq': gen_freq_doc,
}

# Main run loop.
for variant in RUN_VARIANTS:
    cache_path = BASE / f'{variant}.json'
    if cache_path.exists():
        with open(cache_path) as f: n = len(json.load(f))
        print(f'\n{variant}: cached ({n})'); continue

    print(f'\n{"="*60}\n{variant}: {DOCS_PER_VARIANT} docs × 1 target slots\n{"="*60}')

    rng = np.random.default_rng(SEED)
    gen = GEN[variant]
    t0 = time.time()
    results = []

    for doc_idx in tqdm(range(DOCS_PER_VARIANT), desc=variant):
        full_ids = gen(rng)
        n_tok = len(full_ids)
        rem_start = MAX_CTX
        rem_end = n_tok - TARGET_LEN
        for frac in TARGET_FRACS:
            ts = int(rem_start + frac * (rem_end - rem_start))
            te = ts + TARGET_LEN
            if ts - MAX_CTX < 0 or te > n_tok: continue
            r = compute_longrange_curves(full_ids, ts, te)
            r['corpus_id'] = variant
            r['document_id'] = f'{variant}__doc{doc_idx:03d}'
            r['target_id'] = f'{variant}__doc{doc_idx:03d}__pos{int(frac*100):02d}'
            r['target_frac'] = frac
            results.append(r)

    elapsed = time.time() - t0
    with open(cache_path, 'w') as f:
        json.dump(results, f)
    print(f'  {len(results)} results in {elapsed/60:.1f} min')

    if results:
        ord_long = float(np.mean([r['ordered_ppl'][-1] for r in results]))
        shuf_long = float(np.mean([r['shuffled_ppl'][-1] for r in results]))
        ord_zero = float(np.mean([r['ordered_ppl'][0] for r in results]))
        print(f'  ord_ppl[ctx=0]={ord_zero:.2f}  ord_ppl[ctx={MAX_CTX}]={ord_long:.2f}  '
              f'shuf_ppl[ctx={MAX_CTX}]={shuf_long:.2f}  gap@{MAX_CTX}={shuf_long-ord_long:+.2f}')

print('\nDone.')